# 021 — Disaggregation post-processing (IML-based)

Turns the **IML-based** seismic-hazard disaggregations run by
`020-disaggregation.ipynb` into flat, provenance-tracked pickles for record
selection.

For each `(IM definition, epsilon truncation)` pair the notebook:

1. Resolves the OpenQuake `calc_id` of every target-IML disaggregation from
   `wp1/disagg_manifest.json` — no hardcoded integers.
2. Collates them into **one flat dictionary keyed by site index**,
   `disagg_data[site][imt][iml] -> DataFrame` (the disaggregation results are
   **not** grouped by seismicity/region).
3. Builds a flat disagg-**stats** DataFrame, one row per `(site, imt, iml)`.
4. Writes both through the project's provenance cache (`cache_utils`), so a
   re-run reloads unchanged artifacts and refuses stale ones.

The **occurrence** disaggregation `P(m|X=x)` needs the hazard-curve slope at the
target IML. An `iml_disagg` datastore holds only a single intensity level, so the
slope is taken from the full-resolution hazard-curve pickles produced by
`004-psha_results.ipynb` — after asserting they share the disaggregation runs'
source model / logic tree (`oqhelpers.assert_shared_provenance`).

## Prerequisites

- **`020-disaggregation.ipynb` must have run** (with `DRY_RUN = False`), so every
  `AvgSA_{im}_disagg_eps{eps}_iml{nn}` entry exists in
  `hazard_models/eshm20/wp1/disagg_manifest.json` and its datastore
  (`calc_<id>.hdf5`) is present in the local `oqdata` directory. The datastores
  are machine-local and rebuildable; the manifest is the git-tracked pointer.
- **`004-psha_results.ipynb` must have run** (with `SAVE = True`), producing the
  hazard-curve pickles `AvgSA_{im}_hazard_curves_60sites_{3,4,5}sig.pickle` that
  supply the occurrence slope. Their `eps{N}` ↔ `{N}sig` PSHA calculations
  (`wp1/psha_manifest.json`) must share the disaggregation source model — this is
  asserted at run time.
- Target IMLs come from `data_processed/03_site_hazard/AvgSA_{03,06}_imls_for_disaggregation.csv`
  (written by `017-disagg_imls_for_msa_stripes.ipynb`).

## Dependencies

**Upstream:** `017` (target IMLs) → `020` (disagg runs) and `003`→`004` (PSHA
hazard curves). **Downstream:** the record-selection setup
(`setup_AvgSA0{3,6}_gm_selection.py`) and notebooks `031`–`036` consume the
pickles written here.

The notebook is **data-driven from the manifest**: it processes whatever
`(IM, eps)` pairs are present, so `AvgSA_06` is picked up automatically once its
disaggregations are run (its IML file does not exist yet).

## Output structure

Written to `data_processed/03_site_hazard/`, one `(data, stats)` pair per
`(IM, eps)`:

| File | Object |
|---|---|
| `AvgSA_{im}_disagg_data_wp1sites_eps{eps}.pickle` | `dict[site][imt][iml] -> DataFrame` |
| `AvgSA_{im}_disagg_stats_wp1sites_eps{eps}.pickle` | flat `DataFrame`, one row per `(site, imt, imtl)` |

- `iml` keys are the target level in g at 6 significant figures (`float`).
- Each disaggregation `DataFrame` has columns `TRT, Mag, Dist, Eps, Z,
  P(X>x|T,m), nu_m, P(m|X>x), P(m|X=x)`.
- Each stats row has `site_id, lat, lon, seismicity, region, imt, imtl, poe,
  mafe, rtp` — where `poe` / `mafe` / `rtp` are the annual PoE, mean annual
  frequency of exceedance and return period at `imtl`, interpolated from the
  site's matching-eps hazard curve — plus per-TRT `"<TRT> [%]"` proportions and
  `Mag_mean` / `Dist_mean`.
- A `<file>.manifest.json` provenance sidecar is written next to each pickle.

In [ ]:
%load_ext autoreload
%autoreload 2

## 0. Setup

In [ ]:
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
from openquake.commonlib.datastore import read

from phd_project.config import config
from phd_project.scripts import oq_runner, oqhelpers
from phd_project.scripts import cache_utils

cfg = config.load_config()

In [ ]:
# -----------------------------------------------------------------------------
# PARAMETERS
# -----------------------------------------------------------------------------
HAZ_DIR = cfg["proc_data"]["site_hazard"]
WP1_DIR = cfg["hazard_models"]["eshm20_wp1"]
DISAGG_MANIFEST_FP = cfg["hazard_models"]["eshm20_wp1_disagg_manifest"]
PSHA_MANIFEST_FP = cfg["hazard_models"]["eshm20_wp1_psha_manifest"]
SITES_FP = cfg["results"]["selected_sites_csv"]

# IM definition -> target-IML file (as written by 017). AvgSA 0-6 may not exist yet.
IML_FILES = {
    "03": cfg["proc_data"]["disagg_imls_AvgSA_03"],
    "06": cfg["proc_data"]["disagg_imls_AvgSA_06"],
}

# eps truncation level -> hazard-curve pickle suffix (004 writes _{N}sig).
TRUNC_SUFFIX = {3: "3sig", 4: "4sig", 5: "5sig"}

DISAGG_TYPE = "TRT_Mag_Dist_Eps"
TRADITIONAL = True     # P(m|X>x)
OCCURENCE = True       # P(m|X=x), needs the hazard-curve slope

# FORCE_RECOMPUTE: rebuild every pickle even when its inputs are unchanged.
FORCE_RECOMPUTE = False

print(f"hazard dir:      {HAZ_DIR}")
print(f"disagg manifest: {DISAGG_MANIFEST_FP}")
print(f"psha manifest:   {PSHA_MANIFEST_FP}")
print(f"TRADITIONAL={TRADITIONAL}, OCCURENCE={OCCURENCE}, "
      f"FORCE_RECOMPUTE={FORCE_RECOMPUTE}")

## 1. Load inputs

The manifests, the `{name: calc_id}` map, the target IMLs, and the site metadata
(`results/01_site_selection/sites.csv`, whose row order matches the datastore
site order 0..N-1). The `(IM, eps)` pairs to process are discovered from the
manifest entry names.

In [ ]:
disagg_manifest = oq_runner.load_manifest(DISAGG_MANIFEST_FP)
psha_manifest = oq_runner.load_manifest(PSHA_MANIFEST_FP)
calc_ids = oq_runner.load_calc_ids(DISAGG_MANIFEST_FP)

# Site metadata: sites.csv already carries lat/lon/seismicity/region and is in the
# same row order (0..N-1) as the disaggregation datastores.
site_metadata = pd.read_csv(SITES_FP)
n_sites = len(site_metadata)

# Target IMLs per IM definition (skip an IM whose file is not written yet).
imls_by_im = {}
for im, fp in IML_FILES.items():
    fp = Path(fp)
    if fp.is_file():
        imls_by_im[im] = oq_runner.load_imls(fp)
    else:
        print(f"[skip] AvgSA {im}: no IML file at {fp.name}")

# Discover the (im, eps) pairs present, each an ordered list of (iml_index, name).
_pair_re = re.compile(r"^AvgSA_(?P<im>\d+)_disagg_eps(?P<eps>\d+)_iml(?P<i>\d+)$")
pairs = {}
for name in calc_ids:
    m = _pair_re.match(name)
    if m:
        pairs.setdefault((m["im"], int(m["eps"])), []).append((int(m["i"]), name))
for key in pairs:
    pairs[key].sort()

print(f"{n_sites} sites; {len(pairs)} (IM, eps) pairs present:")
for (im, eps), items in sorted(pairs.items()):
    lo, hi = calc_ids[items[0][1]], calc_ids[items[-1][1]]
    print(f"  AvgSA_{im} eps{eps}: {len(items)} imls (calc {lo}..{hi})")

## 2. Collate and cache the disaggregation data and stats

For each `(IM, eps)` pair: assert the hazard curves share the disaggregation
source model, then collate the per-IML disaggregations into the flat dict and
build the stats DataFrame — each written through `cache_utils.load_or_compute`.

The hazard curves are always needed here: the stats `poe`/`mafe`/`rtp` are read
from them at each IML, and (when `OCCURENCE`) they also supply the occurrence
slope. The provenance fingerprint therefore covers the per-IML disagg configs,
the site model, the target-IML file, the calc ids, and the hazard-curve pickle. A
changed input raises `StaleCacheError`; set `FORCE_RECOMPUTE = True` to overwrite.

In [ ]:
results = {}   # (im, eps) -> {"data": ..., "stats": ...}

for (im, eps), items in sorted(pairs.items()):
    if im not in imls_by_im:
        raise FileNotFoundError(
            f"AvgSA_{im} has disagg calcs but no IML file {IML_FILES[im]}")

    names = [name for _, name in items]
    pair_calc_ids = [calc_ids[name] for name in names]
    imls = [float(imls_by_im[im][i - 1]) for i, _ in items]
    assert len(imls) == len(pair_calc_ids)

    # --- hazard curves (matching eps): supply the stats poe/mafe/rtp at each IML
    # and, when OCCURENCE, the occurrence slope. Assert they share the disagg
    # runs' source model before use.
    oqhelpers.assert_shared_provenance(disagg_manifest, psha_manifest, im, eps)
    hc_fp = HAZ_DIR / f"AvgSA_{im}_hazard_curves_60sites_{TRUNC_SUFFIX[eps]}.pickle"
    with open(hc_fp, "rb") as f:
        hcurves = pickle.load(f)

    data_fp = HAZ_DIR / f"AvgSA_{im}_disagg_data_wp1sites_eps{eps}.pickle"
    stats_fp = HAZ_DIR / f"AvgSA_{im}_disagg_stats_wp1sites_eps{eps}.pickle"

    # --- provenance fingerprint (shared by the data and stats artifacts)
    fp_inputs = {f"config_iml{i:02d}": WP1_DIR / oq_runner.disagg_config_name(im, eps, i)
                 for i, _ in items}
    fp_inputs["sites_csv"] = SITES_FP
    fp_inputs["iml_csv"] = IML_FILES[im]
    fp_inputs["calc_ids"] = str(pair_calc_ids)
    fp_inputs["hazard_curves"] = hc_fp
    fp_dict = cache_utils.fingerprint(**fp_inputs)

    disagg_data = cache_utils.load_or_compute(
        data_fp, fp_dict,
        lambda: oqhelpers.collate_disagg_by_iml(
            pair_calc_ids, imls, disagg_type=DISAGG_TYPE,
            traditional=TRADITIONAL, occurence=OCCURENCE,
            hcurves=hcurves, reader=read),
        force_recompute=FORCE_RECOMPUTE)

    disagg_stats = cache_utils.load_or_compute(
        stats_fp, fp_dict,
        lambda: oqhelpers.get_iml_disagg_stats(disagg_data, site_metadata, hcurves),
        force_recompute=FORCE_RECOMPUTE)

    results[(im, eps)] = {"data": disagg_data, "stats": disagg_stats}
    print(f"AvgSA_{im} eps{eps}: {data_fp.name} + {stats_fp.name} "
          f"({len(disagg_stats)} stat rows)")

## 3. Sanity checks

Confirm the flat structure, the per-IML keys, the DataFrame columns, and that the
TRT proportions sum to ~100 % per stats row.

In [ ]:
(im, eps) = sorted(results)[0]
data = results[(im, eps)]["data"]
stats = results[(im, eps)]["stats"]

site0 = sorted(data)[0]
imt0 = list(data[site0])[0]
imls_present = sorted(data[site0][imt0])
print(f"AvgSA_{im} eps{eps}: {len(data)} sites keyed 0..{max(data)}")
print(f"  site {site0}, imt {imt0!r}: {len(imls_present)} imls -> {imls_present}")

df0 = data[site0][imt0][imls_present[0]]
print("  disagg df columns:", list(df0.columns))
print("  stats columns:    ", list(stats.columns))

pct_cols = [c for c in stats.columns if c.endswith("[%]")]
print("  TRT % row-sum summary:")
print(stats[pct_cols].sum(axis=1).describe()[["min", "mean", "max"]].to_string())